In [2]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "tfm-mercado-laboral-datos"
client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT *
FROM `tfm-mercado-laboral-datos.mercado_laboral_datos.vista_maestra_mercado_laboral`
"""
df = client.query(query).to_dataframe()

print(df.shape)
df.head()

(2163, 26)


,job_id,title,categoria_rol,description,skills_desc,formatted_work_type,formatted_experience_level,remote_allowed,location,views,...,employee_count,follower_count,skills_lista,industrias_lista,beneficios_lista,min_salary,med_salary,max_salary,pay_period,currency
0,3245063922,Data Architect,Data Architect,Request: Data ArchitectLocation: San Francisco...,None,Contract,None,NaN,"San Francisco, CA",7.0,...,304,125864,"Engineering, Information Technology",IT Services and IT Consulting,None,NaN,NaN,NaN,None,None
1,3742692445,Sr Data Engineer with Kafka,Data Engineer,Data Engineer with Kafka (W2 Only)💯% Remote\nM...,None,Full-time,None,1.0,"Austin, TX",39.0,...,3,51,None,IT Services and IT Consulting,None,NaN,NaN,NaN,None,None
2,3780415828,Technical Business Analyst,Business Analyst,We’re actively seeking a Technical Business An...,None,Contract,None,NaN,"Albany, NY",4.0,...,43,807,"Business Development, Sales",Financial Services,None,60000.0,NaN,80000.0,YEARLY,USD
3,3797449314,Cloud Platform/ Big Data Engineer,Data Engineer,About Subaru Research and Development:Do you c...,None,Full-time,Entry level,NaN,"Michigan, United States",7.0,...,76,1093,Engineering,"Motor Vehicle Manufacturing, IT Services and I...","Medical insurance, Vision insurance, Dental in...",NaN,NaN,NaN,None,None
4,3800272386,Data Science Software Engineer,Data Scientist,Company Overview Sovrinti is a full-service en...,None,Full-time,None,1.0,United States,125.0,...,2,39,None,IT Services and IT Consulting,"Vision insurance, Medical insurance, Dental in...",NaN,NaN,NaN,None,None


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2163 entries, 0 to 2162
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   job_id                      2163 non-null   Int64  
 1   title                       2163 non-null   object 
 2   categoria_rol               2163 non-null   object 
 3   description                 2163 non-null   object 
 4   skills_desc                 25 non-null     object 
 5   formatted_work_type         2163 non-null   object 
 6   formatted_experience_level  1532 non-null   object 
 7   remote_allowed              643 non-null    float64
 8   location                    2163 non-null   object 
 9   views                       2130 non-null   float64
 10  applies                     1196 non-null   float64
 11  fecha_publicacion           2163 non-null   dbdate 
 12  empresa                     2141 non-null   object 
 13  company_size                2058 

In [4]:
df['remote_allowed'].value_counts(dropna=False)

,count
remote_allowed,
NaN,1520
1.0,643


In [5]:
df['modalidad_remoto'] = df['remote_allowed'].map({1.0: 'Remoto'}).fillna('No especificado')
df['modalidad_remoto'].value_counts()

,count
modalidad_remoto,
No especificado,1520
Remoto,643


In [6]:
df['formatted_experience_level'].value_counts(dropna=False)

,count
formatted_experience_level,
Mid-Senior level,1040
None,631
Entry level,275
Associate,151
Director,32
Internship,27
Executive,7


In [7]:
df['formatted_experience_level'] = df['formatted_experience_level'].fillna('No especificado')
df['formatted_experience_level'].value_counts()

,count
formatted_experience_level,
Mid-Senior level,1040
No especificado,631
Entry level,275
Associate,151
Director,32
Internship,27
Executive,7


In [8]:
df['pay_period'].value_counts(dropna=False)

,count
pay_period,
None,1498
YEARLY,457
HOURLY,203
MONTHLY,5


In [9]:
    factor_anual = {'YEARLY': 1, 'MONTHLY': 12, 'HOURLY': 2080}
df['factor_anual'] = df['pay_period'].map(factor_anual)

df['salario_min_anual'] = df['min_salary'] * df['factor_anual']
df['salario_max_anual'] = df['max_salary'] * df['factor_anual']

df[['pay_period', 'min_salary', 'factor_anual', 'salario_min_anual']].dropna().head(10)

,pay_period,min_salary,factor_anual,salario_min_anual
2,YEARLY,60000.0,1.0,60000.0
7,HOURLY,101.0,2080.0,210080.0
9,YEARLY,150000.0,1.0,150000.0
13,YEARLY,95000.0,1.0,95000.0
24,YEARLY,60000.0,1.0,60000.0
27,YEARLY,61261.0,1.0,61261.0
30,YEARLY,100000.0,1.0,100000.0
32,YEARLY,88854.0,1.0,88854.0
42,YEARLY,77265.0,1.0,77265.0
43,HOURLY,35.0,2080.0,72800.0


In [10]:
df[['salario_min_anual', 'salario_max_anual']].describe()

,salario_min_anual,salario_max_anual
count,622.000000,6.220000e+02
mean,113876.646158,1.775830e+05
std,42871.019370,5.458220e+05
min,30.000000,3.500000e+01
25%,83200.000000,1.144000e+05
50%,110000.000000,1.456000e+05
75%,140000.000000,1.872000e+05
max,400000.000000,1.366560e+07


In [11]:
df[(df['salario_min_anual'] < 1000) | (df['salario_max_anual'] > 500000)][
    ['title', 'pay_period', 'min_salary', 'max_salary', 'salario_min_anual', 'salario_max_anual']
]

,title,pay_period,min_salary,max_salary,salario_min_anual,salario_max_anual
142,Java-Big Data Architect,YEARLY,75.0,85.0,75.0,85.0
215,"ML Engineer L5, LLM Application Frameworks, Ma...",YEARLY,100000.0,720000.0,100000.0,720000.0
279,Vice President of Data Engineering,YEARLY,400000.0,600000.0,400000.0,600000.0
542,Information Technology Business Analyst,YEARLY,60.0,65.0,60.0,65.0
645,Data Engineer,HOURLY,65.0,6570.0,135200.0,13665600.0
662,Business Analyst 24-02965,YEARLY,30.0,35.0,30.0,35.0
873,Sr. Data Engineer,YEARLY,70.0,80.0,70.0,80.0
1091,Data Architect,YEARLY,70.0,75.0,70.0,75.0
1444,Full Time - Financial Business Analyst with Re...,YEARLY,50.0,55.0,50.0,55.0
1611,Finance Senior Oracle Fusion Business Analyst,YEARLY,150.0,150.0,150.0,150.0


In [12]:
df.loc[(df['salario_min_anual'] < 1000) | (df['salario_min_anual'] > 1000000), 'salario_min_anual'] = None
df.loc[(df['salario_max_anual'] < 1000) | (df['salario_max_anual'] > 1000000), 'salario_max_anual'] = None

df[['salario_min_anual', 'salario_max_anual']].describe()

,salario_min_anual,salario_max_anual
count,614.000000,613.000000
mean,115359.485195,157896.340538
std,41117.803429,65093.008124
min,37315.200000,47840.000000
25%,84300.000000,114400.000000
50%,110000.000000,145600.000000
75%,141080.000000,187200.000000
max,400000.000000,720000.000000


In [14]:
import plotly.express as px

conteo_roles = df['categoria_rol'].value_counts().reset_index()
conteo_roles.columns = ['categoria_rol', 'num_ofertas']

fig = px.bar(
    conteo_roles.sort_values('num_ofertas'),
    x='num_ofertas',
    y='categoria_rol',
    orientation='h',
    text='num_ofertas',
    title='Distribución de ofertas por categoría de rol'
)

fig.update_traces(marker_color='#2a78d6', textposition='outside')
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='Número de ofertas',
    yaxis_title='',
    font=dict(color='#0b0b0b', size=13),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_yaxes(showgrid=False)
fig.update_layout(margin=dict(l=180))
fig.show()

In [17]:
import pandas as pd

In [18]:
orden_experiencia = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive', 'No especificado']
orden_roles = conteo_roles.sort_values('num_ofertas', ascending=False)['categoria_rol'].tolist()

tabla_cruzada = pd.crosstab(df['categoria_rol'], df['formatted_experience_level'])
tabla_cruzada = tabla_cruzada.reindex(index=orden_roles, columns=orden_experiencia)

import plotly.graph_objects as go

colorscale_azul = [
    [0.0, '#cde2fb'],
    [0.25, '#86b6ef'],
    [0.5, '#3987e5'],
    [0.75, '#1c5cab'],
    [1.0, '#0d366b'],
]

fig = go.Figure(data=go.Heatmap(
    z=tabla_cruzada.values,
    x=tabla_cruzada.columns,
    y=tabla_cruzada.index,
    colorscale=colorscale_azul,
    colorbar=dict(title='Nº ofertas'),
    hovertemplate='%{y} - %{x}: %{z} ofertas<extra></extra>',
))

fig.update_layout(
    title='Categoría de rol x Nivel de experiencia (nº de ofertas)',
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    font=dict(color='#0b0b0b', size=13),
    margin=dict(l=180),
)
fig.show()

In [19]:
evolucion = df.groupby('fecha_publicacion').size().reset_index(name='num_ofertas')
evolucion = evolucion.sort_values('fecha_publicacion')

fig = px.line(
    evolucion,
    x='fecha_publicacion',
    y='num_ofertas',
    title='Evolución diaria de ofertas publicadas (datos y analytics)',
    markers=True,
)

fig.update_traces(line_color='#2a78d6', marker=dict(color='#2a78d6', size=7))
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='Fecha de publicación',
    yaxis_title='Número de ofertas',
    font=dict(color='#0b0b0b', size=13),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_yaxes(gridcolor='#e1e0d9', zeroline=False)
fig.show()

In [20]:
evolucion['dia_semana'] = evolucion['fecha_publicacion'].apply(lambda f: f.strftime('%A'))
evolucion

,fecha_publicacion,num_ofertas,dia_semana
0,2024-04-05,66,Friday
1,2024-04-06,98,Saturday
2,2024-04-07,6,Sunday
3,2024-04-09,148,Tuesday
4,2024-04-11,189,Thursday
5,2024-04-12,82,Friday
6,2024-04-15,135,Monday
7,2024-04-16,99,Tuesday
8,2024-04-17,139,Wednesday
9,2024-04-18,680,Thursday


In [21]:
evolucion['fecha_publicacion'] = pd.to_datetime(evolucion['fecha_publicacion'])
evolucion_num = evolucion[['fecha_publicacion', 'num_ofertas']]

rango_completo = pd.date_range(start='2024-04-05', end='2024-04-20', freq='D')
evolucion_completa = evolucion_num.set_index('fecha_publicacion').reindex(rango_completo, fill_value=0)
evolucion_completa.index.name = 'fecha_publicacion'
evolucion_completa = evolucion_completa.reset_index()
evolucion_completa['dia_semana'] = evolucion_completa['fecha_publicacion'].dt.strftime('%A')

evolucion_completa

,fecha_publicacion,num_ofertas,dia_semana
0,2024-04-05,66,Friday
1,2024-04-06,98,Saturday
2,2024-04-07,6,Sunday
3,2024-04-08,0,Monday
4,2024-04-09,148,Tuesday
5,2024-04-10,0,Wednesday
6,2024-04-11,189,Thursday
7,2024-04-12,82,Friday
8,2024-04-13,0,Saturday
9,2024-04-14,0,Sunday


In [22]:
fig = px.line(
    evolucion_completa,
    x='fecha_publicacion',
    y='num_ofertas',
    title='Evolución diaria de ofertas publicadas (datos y analytics)',
    markers=True,
)

fig.update_traces(line_color='#2a78d6', marker=dict(color='#2a78d6', size=7))
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='Fecha de publicación',
    yaxis_title='Número de ofertas',
    font=dict(color='#0b0b0b', size=13),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_yaxes(gridcolor='#e1e0d9', zeroline=False)
fig.show()

In [24]:
tabla_remoto = pd.crosstab(df['categoria_rol'], df['modalidad_remoto'], normalize='index') * 100
tabla_remoto = tabla_remoto.reindex(index=orden_roles[::-1])
tabla_remoto = tabla_remoto[['Remoto', 'No especificado']]

fig = go.Figure()
fig.add_bar(
    y=tabla_remoto.index,
    x=tabla_remoto['Remoto'],
    name='Remoto',
    orientation='h',
    marker_color='#2a78d6',
)
fig.add_bar(
    y=tabla_remoto.index,
    x=tabla_remoto['No especificado'],
    name='No especificado',
    orientation='h',
    marker_color='#c3c2b7',
)

fig.update_layout(
    barmode='stack',
    title='Modalidad remota por categoría de rol (% de ofertas)',
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='% de ofertas',
    yaxis_title='',
    font=dict(color='#0b0b0b', size=13),
    margin=dict(l=180),
    legend=dict(orientation='h', y=1.1),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False, range=[0, 100])
fig.update_yaxes(showgrid=False)
fig.show()

In [25]:
df.groupby('categoria_rol')['salario_min_anual'].count().reindex(orden_roles)

,salario_min_anual
categoria_rol,
Business Analyst,135
Data Analyst,113
Data Engineer,119
Data Scientist,109
BI Analyst / Developer,61
Machine Learning Engineer,49
Data Architect,28


In [26]:
df['salario_estimado_anual'] = df[['salario_min_anual', 'salario_max_anual']].mean(axis=1)

fig = px.box(
    df,
    x='categoria_rol',
    y='salario_estimado_anual',
    category_orders={'categoria_rol': orden_roles},
    title='Distribución del salario anual estimado por categoría de rol',
)

fig.update_traces(marker_color='#2a78d6', line_color='#2a78d6')
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='',
    yaxis_title='Salario anual estimado ($)',
    font=dict(color='#0b0b0b', size=13),
)
fig.update_yaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_xaxes(tickangle=-30)
fig.show()

In [27]:
df['industrias_lista'].head(10)

,industrias_lista
0,IT Services and IT Consulting
1,IT Services and IT Consulting
2,Financial Services
3,"Motor Vehicle Manufacturing, IT Services and I..."
4,IT Services and IT Consulting
5,IT Services and IT Consulting
6,"Technology, Information and Internet"
7,None
8,"Technology, Information and Internet"
9,Software Development


In [28]:
industrias_expandido = df['industrias_lista'].dropna().str.split(', ').explode()
conteo_industrias = industrias_expandido.value_counts().head(15)
conteo_industrias

,count
industrias_lista,
IT Services and IT Consulting,635
Financial Services,276
Software Development,266
Technology,126
Hospitals and Health Care,112
Staffing and Recruiting,103
Insurance,93
Information Services,89
Information and Media,77


In [29]:
top_industrias = conteo_industrias.head(10).reset_index()
top_industrias.columns = ['industria', 'num_ofertas']

fig = px.bar(
    top_industrias.sort_values('num_ofertas'),
    x='num_ofertas',
    y='industria',
    orientation='h',
    text='num_ofertas',
    title='Top 10 industrias con más ofertas de datos y analytics',
)

fig.update_traces(marker_color='#2a78d6', textposition='outside')
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='Número de ofertas (una oferta puede contar en varias industrias)',
    yaxis_title='',
    font=dict(color='#0b0b0b', size=13),
    margin=dict(l=230),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_yaxes(showgrid=False)
fig.show()

In [30]:
df[['views', 'applies']].describe()

,views,applies
count,2130.000000,1196.000000
mean,47.857746,25.602007
std,105.385432,49.317149
min,1.000000,1.000000
25%,4.000000,2.000000
50%,9.000000,8.000000
75%,43.000000,27.000000
max,1393.000000,566.000000


In [31]:
correlacion = df[['views', 'applies']].corr(method='spearman')
print(correlacion)

            views   applies
views    1.000000  0.929871
applies  0.929871  1.000000


In [32]:
fig = px.scatter(
    df.dropna(subset=['views', 'applies']),
    x='views',
    y='applies',
    title='Relación entre vistas y solicitudes por oferta',
    log_x=True,
    log_y=True,
    opacity=0.5,
)

fig.update_traces(marker=dict(color='#2a78d6', size=6))
fig.update_layout(
    plot_bgcolor='#fcfcfb',
    paper_bgcolor='#fcfcfb',
    xaxis_title='Vistas (escala logarítmica)',
    yaxis_title='Solicitudes (escala logarítmica)',
    font=dict(color='#0b0b0b', size=13),
)
fig.update_xaxes(gridcolor='#e1e0d9', zeroline=False)
fig.update_yaxes(gridcolor='#e1e0d9', zeroline=False)
fig.show()

In [33]:
from scipy.stats import mannwhitneyu

salario_ba = df[df['categoria_rol'] == 'Business Analyst']['salario_estimado_anual'].dropna()
salario_mle = df[df['categoria_rol'] == 'Machine Learning Engineer']['salario_estimado_anual'].dropna()

estadistico, p_valor = mannwhitneyu(salario_ba, salario_mle, alternative='two-sided')

print(f"Business Analyst: n={len(salario_ba)}, mediana={salario_ba.median():.0f}$")
print(f"Machine Learning Engineer: n={len(salario_mle)}, mediana={salario_mle.median():.0f}$")
print(f"Estadístico U = {estadistico:.1f}")
print(f"p-valor = {p_valor:.6f}")

Business Analyst: n=135, mediana=105000$
Machine Learning Engineer: n=49, mediana=182000$
Estadístico U = 607.5
p-valor = 0.000000


In [34]:
print(f"p-valor = {p_valor:.2e}")

p-valor = 2.82e-17


In [35]:
salario_da = df[df['categoria_rol'] == 'Data Analyst']['salario_estimado_anual'].dropna()

estadistico2, p_valor2 = mannwhitneyu(salario_ba, salario_da, alternative='two-sided')

print(f"Business Analyst: n={len(salario_ba)}, mediana={salario_ba.median():.0f}$")
print(f"Data Analyst: n={len(salario_da)}, mediana={salario_da.median():.0f}$")
print(f"Estadístico U = {estadistico2:.1f}")
print(f"p-valor = {p_valor2:.2e}")

Business Analyst: n=135, mediana=105000$
Data Analyst: n=113, mediana=98800$
Estadístico U = 8121.5
p-valor = 3.80e-01
